# Lección 2.8 — La maldición de la dimensionalidad (intuición)

> **Módulo 2 · Los datos: el combustible** · Nivel 1 — Cero
> ⏱️ Tiempo estimado: 80–100 min (lección con ejercicios intercalados ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lecciones 2.3 (escalado y distancias), 2.5 (división de datos) y 2.6 (pipelines)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía por qué agregar columnas puede **empeorar** un modelo.
- Describir tres síntomas de la maldición: el **espacio vacío**, las **distancias que se parecen** y el **sobreajuste** con más columnas que filas.
- Reconocer qué modelos la sufren más (los basados en distancias, como KNN).
- Aplicar remedios: **seleccionar** columnas (dentro de un pipeline), conseguir más datos, reducir dimensiones.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_digits, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: buscar a un amigo

Quedaste de verte con un amigo, pero no sabes exactamente dónde está:
- **En una calle** (1 dimensión): caminas de un extremo al otro y lo encuentras.
- **En un parque** (2 dimensiones): ya tienes que recorrer a lo largo **y** a lo ancho. Tardas mucho más.
- **En un edificio de 20 pisos** (3 dimensiones): largo, ancho **y** altura. Mucho más difícil.

Cada **dimensión** nueva multiplica el espacio donde buscar. En ML, **cada columna es una dimensión**. Una tabla con 100 columnas describe puntos en un espacio de 100 dimensiones: imposible de imaginar, y con propiedades muy raras.

La **maldición de la dimensionalidad** es el nombre que se le da a todos los problemas que aparecen cuando hay **muchas columnas** comparadas con la cantidad de **filas**. La idea central:

> **Con cada columna nueva, tus datos quedan más "desparramados", y necesitas muchísimos más ejemplos para cubrir el espacio.**

### ✋ Ejercicio 1 — Calentamiento
Para que un modelo aprenda bien, quieres tener al menos **un ejemplo** en cada "casilla" del espacio. Si divides cada columna en **10 tramos** (por ejemplo, edad en 10 grupos), ¿cuántas casillas hay con 1 columna? ¿Con 2? ¿Con 3? ¿Con 10?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- 1 columna: **10** casillas · 2: **100** (10 × 10) · 3: **1,000** · 10: **10,000,000,000** (diez mil millones).

Con 10 columnas necesitarías **diez mil millones** de ejemplos para tener uno por casilla. Nadie tiene eso. Con 100 columnas el número tiene 100 ceros. **Los datos nunca alcanzan para llenar un espacio de muchas dimensiones**: casi todas las casillas están vacías.
</details>

---
## 2. 📖 Síntoma 1: el vecindario deja de ser "vecino"

Imagina datos repartidos de forma pareja dentro de un cubo. Quieres el **10 % de los datos más cercanos** a un punto (sus "vecinos"). ¿Qué tan grande tiene que ser el "cubito" que los contiene?

### 🔮 Ejercicio 2 — ¿Qué tan grande es un vecindario?
En **1 dimensión** (una recta de largo 1), para atrapar el 10 % de los datos basta un tramo de largo **0.1**. Antes de ejecutar: en **10 dimensiones**, ¿qué largo tendrá cada lado del cubito que atrapa el 10 % de los datos?

- a) 0.1 · b) 0.3 · c) 0.8 · d) 1.0

✍️ *Mi apuesta:*

In [ ]:
dimensiones = [1, 2, 3, 5, 10, 50, 100]
for d in dimensiones:
    lado = 0.10 ** (1 / d)          # lado del cubito que contiene el 10 % del volumen
    barra = "█" * int(lado * 40)
    print(f"{d:>3} dimensiones → lado = {lado:.2f}  {barra}")

<details><summary><b>👀 Ver respuesta</b></summary>

**c) ≈0.79.** En 10 dimensiones, para atrapar apenas el 10 % de los datos, el cubito debe abarcar **el 79 % del rango de cada columna**. En 100 dimensiones, el 98 %.

¿Qué significa? Que los "vecinos más cercanos" **ya no están cerca**: para encontrar unos cuantos, hay que ir casi de un extremo al otro en cada columna. Un modelo como KNN (Lección 5.3), que asume "los parecidos están cerca", pierde su fundamento.
</details>

---
## 3. 📖 Síntoma 2: todos están igual de lejos

Hay algo aún más extraño. Toma un punto al azar y mide su distancia a otros 1,000 puntos al azar. En pocas dimensiones, algunos estarán **muy cerca** y otros **muy lejos**. Pero a medida que crecen las dimensiones…

### 🔮 Ejercicio 3 — El vecino más cercano y el más lejano
Antes de ejecutar: al aumentar las dimensiones, ¿qué pasa con la diferencia entre el vecino **más cercano** y el **más lejano**?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(0)
contraste = []
dims = [1, 2, 5, 10, 50, 100, 500, 1000]
for d in dims:
    puntos = rng.random((1_000, d))
    referencia = rng.random(d)
    distancias = np.sqrt(((puntos - referencia) ** 2).sum(axis=1))
    contraste.append(distancias.max() / distancias.min())
    print(f"{d:>5} dimensiones → el más lejano está {distancias.max() / distancias.min():>9.1f} veces más lejos que el más cercano")

plt.semilogx(dims, contraste, "o-"); plt.yscale("log")
plt.xlabel("número de dimensiones (columnas)"); plt.ylabel("más lejano / más cercano")
plt.title("En muchas dimensiones, todos los puntos están casi igual de lejos"); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La diferencia **se desploma**. En 1 dimensión, el más lejano está ≈2,600 veces más lejos que el más cercano. En 1,000 dimensiones, apenas ≈**1.1 veces**: **todos los puntos están casi a la misma distancia**.

Si todos están igual de lejos, "el más cercano" es casi una **lotería**. Por eso los modelos basados en distancias (KNN, y en parte SVM y el clustering del Módulo 8) son los que más sufren esta maldición.
</details>

---
## 4. 📖 Qué pasa si… agregas columnas inútiles

En la vida real, la maldición casi siempre llega así: alguien agrega **muchas columnas "por si acaso"** (Lección 2.4), y la mayoría no tienen relación con lo que se quiere predecir. Son **ruido**.

Vamos a tomar el problema de las dos lunas (2 columnas útiles) y agregarle columnas de ruido.

### 🔮 Ejercicio 4 — KNN contra árbol
Antes de ejecutar: al agregar 10, 50 y 200 columnas de ruido, ¿qué modelo crees que pierde **más** exactitud, el KNN o el árbol de decisión? ¿Por qué?

✍️ *Mi apuesta:*

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=500, noise=0.25, random_state=0)
rng = np.random.default_rng(1)

print("columnas de ruido   KNN      árbol")
for extra in [0, 10, 50, 200]:                         # ✏️ prueba 500
    ruido = rng.normal(size=(len(X_lunas), extra))
    X_con_ruido = np.hstack([X_lunas, ruido])
    knn = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
                          X_con_ruido, y_lunas, cv=5).mean()
    arbol = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=0), X_con_ruido, y_lunas, cv=5).mean()
    print(f"{extra:>10}          {knn:.1%}    {arbol:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El **KNN** se derrumba: de ≈**93 %** sin ruido a ≈**69 %** con 200 columnas de ruido (y ✏️ con 500 se acerca todavía más a una moneda al aire). El **árbol** resiste mucho mejor: baja de ≈90 % a ≈86 %.

¿Por qué? El KNN mide distancias usando **todas** las columnas por igual: las 2 útiles quedan ahogadas entre las de ruido (Síntoma 2). El árbol, en cambio, **elige** en cada pregunta la columna que mejor divide, así que casi siempre usa las útiles e ignora el ruido. No es inmune (con más ruido, alguna columna inútil "parece" útil por casualidad), pero sufre mucho menos.
</details>

---
## 5. 📖 Síntoma 3: más columnas que filas

Cuando hay **más columnas que ejemplos**, el modelo tiene tanta libertad que puede encontrar una combinación que **explica perfectamente** el entrenamiento… aunque los datos sean puro ruido. Es como tener 100 perillas para ajustar 50 puntos: siempre hay una configuración que los "acierta" todos.

Esto es frecuente en medicina (pocos pacientes, miles de mediciones genéticas) y en texto (pocos documentos, miles de palabras, Lección 2.1).

### 🔮 Ejercicio 5 — 50 pacientes, 500 mediciones
Creamos 50 pacientes con 500 mediciones **al azar** y un diagnóstico **al azar**. Antes de ejecutar: ¿qué exactitud tendrá una regresión logística en **entrenamiento**? ¿Y con 50 pacientes **nuevos**?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(2)
COLUMNAS = 500                                         # ✏️ prueba 5, 20, 50
X_gen, y_gen = rng.normal(size=(50, COLUMNAS)), rng.integers(0, 2, 50)
X_gen_nuevos, y_gen_nuevos = rng.normal(size=(50, COLUMNAS)), rng.integers(0, 2, 50)

modelo = LogisticRegression(C=1_000, max_iter=5_000).fit(X_gen, y_gen)
print(f"Entrenamiento   : {modelo.score(X_gen, y_gen):.0%}")
print(f"Pacientes nuevos: {modelo.score(X_gen_nuevos, y_gen_nuevos):.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**100 % en entrenamiento** y ≈**50 %** con pacientes nuevos (lo que corresponde a datos al azar). Con 500 columnas y 50 filas, el modelo puede "explicar" cualquier cosa, **incluso el ruido**: es sobreajuste extremo (Lección 4.4).

✏️ Si bajas a `COLUMNAS = 5`, el entrenamiento ya no llega al 100 %: con pocas perillas, el modelo no puede memorizar 50 etiquetas al azar.

**Regla práctica:** desconfía cuando el número de columnas se acerca al de filas o lo supera.
</details>

---
## 6. 📖 La buena noticia: los datos reales no llenan todo el espacio

Si la maldición fuera tan terrible siempre, nadie podría clasificar imágenes (¡una foto pequeña tiene miles de píxeles!). La razón por la que sí se puede:

> Los datos reales **no** están repartidos al azar por todo el espacio. Viven en una **parte pequeña** y ordenada de él.

Piensa en los dígitos escritos a mano de la Lección 2.1: tienen 64 píxeles, pero no cualquier combinación de 64 grises es un dígito. Los dígitos reales ocupan una "rebanada" diminuta de ese espacio de 64 dimensiones, y dentro de ella los parecidos sí están cerca.

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
print(f"Dígitos: {X_dig.shape[0]} imágenes · {X_dig.shape[1]} columnas (píxeles)")
exactitud = cross_val_score(KNeighborsClassifier(n_neighbors=3), X_dig, y_dig, cv=5).mean()
print(f"KNN con 64 dimensiones: {exactitud:.1%}  (adivinando al azar: 10%)")

### ✋ Ejercicio 6
El KNN funciona muy bien con 64 dimensiones en los dígitos, pero se derrumbó con 202 dimensiones en las lunas con ruido. ¿Cuál es la diferencia clave entre los dos casos?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En los dígitos (≈**97 %** de exactitud), **todas** las columnas están relacionadas con lo que importa: los píxeles cercanos se parecen entre sí y juntos dibujan la forma del número. Los datos viven en una estructura ordenada. En las lunas, **200 de las 202 columnas eran ruido puro**, sin relación con nada: llenaban el espacio al azar y ahogaban a las 2 útiles.

**Lo que hace daño no es tener muchas columnas, sino tener muchas columnas que no aportan.** Esta es la idea detrás de la reducción de dimensionalidad (PCA, Lección 8.4): encontrar esa "rebanada" donde viven los datos.
</details>

---
## 7. 📖 Remedios

| Remedio | Idea | Dónde lo verás |
|---|---|---|
| **Pensar antes de agregar** | Cada columna debe tener una razón de ser | Lección 2.4 |
| **Seleccionar columnas** | Quedarse con las más relacionadas con la etiqueta (`SelectKBest`…) | Esta lección |
| **Más datos** | Llenar mejor el espacio | — |
| **Regularización** | Castigar al modelo por usar demasiadas perillas | Lección 7.1 |
| **Reducir dimensiones** | Resumir muchas columnas en pocas que capturan lo esencial (PCA) | Lección 8.4 |
| **Modelos que eligen columnas** | Árboles y ensambles ignoran mejor las columnas inútiles | Lección 5.4, Módulo 6 |

### 🔮 Ejercicio 7 — Rescatar al KNN
Volvamos a las lunas con 200 columnas de ruido. Esta vez pondremos un **selector** que se queda con las `K` columnas más relacionadas con la etiqueta **antes** del KNN, todo **dentro de un pipeline** (¡para no repetir la fuga de la Lección 2.6!). Antes de ejecutar: ¿qué `K` crees que funcionará mejor?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(1)
X_ruidosas = np.hstack([X_lunas, rng.normal(size=(len(X_lunas), 200))])

for K in [2, 5, 20, 100, 202]:                         # ✏️
    tuberia = make_pipeline(SelectKBest(f_classif, k=K), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
    print(f"K = {K:>3} columnas → {cross_val_score(tuberia, X_ruidosas, y_lunas, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con **K = 2** el KNN vuelve a ≈**93 %**: el selector encontró las dos columnas útiles entre 202, y el resultado es igual al de los datos sin ruido. Con K = 5 baja un poco (≈89 %); con 20, ≈82 %; y con todas (202) volvemos al desastre: ≈62 %. (No es idéntico al ≈69 % del Ejercicio 4 porque las columnas de ruido se sortearon de nuevo: otro ruido, otro resultado.)

En datos reales no sabrás cuántas columnas son útiles: `K` es un **hiperparámetro** más, que se elige con validación cruzada (Lección 4.6). Y recuerda: el selector va **dentro** del pipeline; si seleccionas con todos los datos antes de validar, el resultado sale inflado (Lección 2.6).
</details>

---
## 8. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué hacer |
|---|---|---|
| Agregaste columnas y el modelo **empeoró** | Columnas de ruido; maldición de la dimensionalidad | Quitar columnas inútiles, seleccionar (en pipeline) |
| Entrenamiento ≈100 %, prueba ≈ azar, con más columnas que filas | Sobreajuste por exceso de dimensiones | Menos columnas, más datos, regularización (7.1) |
| KNN o clustering dan resultados casi aleatorios con muchas columnas | Las distancias dejaron de significar algo | Seleccionar o reducir dimensiones (8.4) |
| One-hot de una columna con miles de categorías | Explosión de columnas (Lección 2.3) | Agrupar raras o *target encoding* |
| Seleccionaste columnas y la validación salió espectacular | Selección hecha con todos los datos (fuga) | Seleccionar dentro del pipeline (Lección 2.6) |

### 🔮 Ejercicio 8 — ¿Ayuda o estorba?
Un equipo tiene 2,000 clientes y 15 columnas bien pensadas. Alguien propone agregar 3,000 columnas: una por cada producto del catálogo (1 si el cliente lo compró alguna vez). ¿Qué riesgos ves? ¿Qué propondrías?

<details><summary><b>👀 Ver respuesta</b></summary>

**Riesgos:** pasarían de 15 a 3,015 columnas con solo 2,000 clientes: **más columnas que filas** (Síntoma 3). La mayoría de esas columnas estarán casi siempre en 0 (cada cliente compró pocos productos), y un KNN quedaría inservible.

**Propuesta:** resumirlas en pocas columnas con significado (Lección 2.4): número de productos distintos comprados, gasto por **categoría** de producto (10–20 categorías en vez de 3,000 productos), producto favorito… O, si se quiere usar todo, un modelo que elija columnas (árboles/ensambles) o una reducción de dimensiones.
</details>

---
## 9. 🧠 Resumen en 5 puntos
1. Cada columna es una **dimensión**; con muchas, el espacio crece tan rápido que los datos **nunca alcanzan** para llenarlo.
2. En muchas dimensiones, los vecinos **ya no están cerca** y **todos los puntos están casi igual de lejos**: los modelos basados en distancias sufren más.
3. Con **más columnas que filas**, un modelo puede "explicar" hasta el ruido: sobreajuste extremo.
4. El daño lo hacen sobre todo las columnas **que no aportan**; los datos reales suelen vivir en una "rebanada" ordenada del espacio.
5. Remedios: pensar antes de agregar, **seleccionar** (dentro de un pipeline), más datos, regularización (7.1), reducción de dimensiones (8.4).

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué modelo sufre **más** cuando agregas muchas columnas de ruido?
- a) Árbol de decisión · b) KNN · c) Los dos igual · d) Ninguno

**2.** Divides cada una de 4 columnas en 10 tramos. ¿Cuántas casillas hay?
- a) 40 · b) 400 · c) 10,000 · d) 1,000,000

**3.** En 1,000 dimensiones con puntos al azar, el vecino más lejano está…
- a) Miles de veces más lejos que el más cercano · b) Casi a la misma distancia que el más cercano · c) Exactamente a la misma distancia · d) No se puede medir

**4.** Tienes 80 pacientes y 10,000 mediciones genéticas. Tu modelo da 100 % en entrenamiento. ¿Qué sospechas?
- a) Un modelo excelente · b) Sobreajuste por exceso de dimensiones · c) Desbalance · d) Falta escalar

**5.** ¿Dónde debe ir el selector de columnas para no causar fuga?
- a) Antes de dividir los datos · b) Dentro del pipeline, junto al modelo · c) Después de evaluar · d) Da igual

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** KNN usa todas las columnas por igual al medir distancias; el árbol elige.
2. **c)** 10 × 10 × 10 × 10 = 10,000.
3. **b)** Las distancias se "concentran": todas se parecen.
4. **b)** Con muchas más columnas que filas, cualquier modelo flexible puede memorizar el entrenamiento. Hay que mirar la validación.
5. **b)** Así, en cada turno de validación, el selector solo ve el entrenamiento (Lección 2.6).
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Si más columnas pueden empeorar el modelo, ¿por qué no usar siempre muy pocas? ¿Qué arriesgas al quitar columnas?

**B. Caso de análisis 💼** Una tienda en línea quiere recomendar productos a cada cliente buscando "clientes parecidos" (KNN) según los 5,000 productos que cada uno ha visto. Sus recomendaciones parecen aleatorias. Explica qué puede estar pasando y propón dos soluciones.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Es otro equilibrio (¡no hay almuerzo gratis!): con muy pocas columnas, puedes dejar fuera información importante y el modelo **no tendrá con qué** acertar (subajuste). Lo que se busca no es "pocas" sino "**las que aportan**", y eso se decide con conocimiento del negocio y con validación.

**B.** Con 5,000 columnas, casi todas en 0 (cada cliente ve pocos productos), las distancias entre clientes **se parecen todas** (Síntoma 2): el "cliente más parecido" es casi una lotería. Soluciones: (1) resumir en categorías o gustos (deportes, tecnología, hogar…) en lugar de productos individuales; (2) reducir dimensiones para encontrar "gustos" ocultos (es la base de los sistemas de recomendación, Lección 8.7); (3) usar medidas de similitud pensadas para estos datos (por ejemplo, contar productos **en común**).
</details>

### 10.3 🧩 Práctica integradora: "Los 200 indicadores"

Una empresa de seguros recopiló **210 indicadores** de 600 clientes para predecir si presentarán un reclamo este año. El equipo sospecha que solo unos pocos sirven de verdad.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
X_seg, y_seg = make_classification(n_samples=600, n_features=210, n_informative=6, n_redundant=0,
                                   n_repeated=0, class_sep=1.2, shuffle=False, random_state=3)
# shuffle=False: las 6 primeras columnas son las útiles (en la vida real no lo sabrías)
X_seg_tr, X_seg_te, y_seg_tr, y_seg_te = train_test_split(X_seg, y_seg, test_size=0.25,
                                                          stratify=y_seg, random_state=0)
print("Entrenamiento:", X_seg_tr.shape, "· Prueba:", X_seg_te.shape)
print(f"Presentaron reclamo: {y_seg.mean():.0%}")

**Paso 2 🔮 — KNN con todo.** Antes de ejecutar: ¿qué exactitud esperas en validación cruzada con los 210 indicadores?

In [ ]:
knn_todo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
print(f"KNN con 210 columnas: {cross_val_score(knn_todo, X_seg_tr, y_seg_tr, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**66 %**, no mucho mejor que adivinar (≈50 %). Las 6 columnas útiles quedan ahogadas por más de 200 de ruido: la maldición en acción.
</details>

**Paso 3 ✏️ — Elige cuántas columnas conservar.** Prueba distintos valores de `K` con el selector **dentro** del pipeline y validación cruzada. ¿Cuál elegirías?

In [ ]:
for K in [3, 6, 10, 20, 50, 100, 210]:                  # ✏️
    tuberia = make_pipeline(SelectKBest(f_classif, k=K), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
    print(f"K = {K:>3} → {cross_val_score(tuberia, X_seg_tr, y_seg_tr, cv=5).mean():.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

El mejor resultado es **K = 6** (≈**94 %**), seguido de K = 10 (≈90 %); a partir de ahí la exactitud cae a medida que se cuelan columnas de ruido. Con K = 3 se quedan fuera columnas útiles (≈80 %). Elegiría **K = 6**.
</details>

**Paso 4 ✋ — ¿Encontró las correctas?** Entrenamos el selector con `K = 6` y vemos qué columnas eligió. En estos datos sabemos que las útiles son las **0, 1, 2, 3, 4 y 5**. ¿Las encontró todas?

In [ ]:
K_ELEGIDO = 6       # ✏️
selector = SelectKBest(f_classif, k=K_ELEGIDO).fit(X_seg_tr, y_seg_tr)
print("Columnas elegidas:", np.flatnonzero(selector.get_support()).tolist())

<details><summary><b>👀 Ver respuesta</b></summary>

Elige **[0, 1, 2, 3, 4, 5]**: ¡las 6 útiles, sin colar ninguna de ruido! No siempre sale tan perfecto: el selector mira **cada columna por separado**, así que con pocos datos alguna columna de ruido puede parecer útil por casualidad, y una útil puede "esconderse" si solo ayuda en combinación con otras. ✏️ Prueba `K_ELEGIDO = 10` y mira cuáles se cuelan.
</details>

**Paso 5 ▶️ — Examen final.** Compara en el conjunto de prueba, una sola vez, el KNN con todas las columnas y el KNN con tu selección.

In [ ]:
final_todo = knn_todo.fit(X_seg_tr, y_seg_tr)
final_sel = make_pipeline(SelectKBest(f_classif, k=K_ELEGIDO), StandardScaler(),
                          KNeighborsClassifier(n_neighbors=15)).fit(X_seg_tr, y_seg_tr)
print(f"KNN con 210 columnas      : {final_todo.score(X_seg_te, y_seg_te):.1%}")
print(f"KNN con {K_ELEGIDO} columnas elegidas: {final_sel.score(X_seg_te, y_seg_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**65 %** con todo contra ≈**95 %** con 6 columnas. Usar **menos** información, bien elegida, dio un modelo **mucho mejor**, más rápido y más fácil de explicar al equipo ("estos 6 indicadores son los que importan").

**Lección del módulo:** los datos son el combustible, pero **más combustible no siempre es mejor: tiene que ser el adecuado**.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "The Curse of Dimensionality" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Feature selection*.
- 📘 **Lectura:** Pedro Domingos, "A Few Useful Things to Know about Machine Learning" (2012), sección "Intuition fails in high dimensions" (inglés).

---
⬅️ **Anterior:** [2.7 Datos desbalanceados: cuando lo importante es raro](2.7-datos-desbalanceados.ipynb) · ➡️ **Siguiente:** [2.9 Proyecto del módulo: Los datos sucios de CineNube 🎬](2.9-proyecto-del-modulo.md)